In [ ]:
# runtime check
!nvidia-smi

In [ ]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# clone repo
!git clone https://github.com/nmeyer19/llama-sft-dpo-lora.git

In [ ]:
# change cd
%cd /content/llama-sft-dpo-lora

In [ ]:
# installs
!pip install -r requirements.txt

In [ ]:
import huggingface_hub

# hf authentication and login
from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")
huggingface_hub.login(token=hf_token)

In [ ]:
# run the evaluation
!PYTHONPATH=. python evaluation/capabilities.py

## Results

In [ ]:
import yaml
import json
import pandas as pd

# load MMLU config
with open("./configs/mmlu.yaml", "r") as file:
    config = yaml.safe_load(file)

# get the locations of the relevant files
gen_results_dir = config["outputs"]["results_dir"]

results = {
    "base-model": gen_results_dir + "/base-model/results.json",
    "sft-model": gen_results_dir + "/sft-model/results.json",
    #"dpo-model": gen_results_dir + "/dpo-model/results.json",
}

# compile results into a single dict
master_dict = {}
for m, r in results.items():
    with open(r, 'r') as file:
        data = json.load(file)
    
    master_dict[m] = data

# construct and display dataframe
subj_df = pd.DataFrame({model: d["per_subject"] for model, d in master_dict.items()})
subj_df.sort_index(inplace=True)
total_df = pd.DataFrame({model: [d["total"]] for model, d in master_dict.items()}, index=["Total"])
output = pd.concat([subj_df, total_df])
display(output.style.highlight_max(axis=1))